In [9]:
import numpy as np
import keras
from sklearn.feature_selection import SelectKBest
from sklearn.feature_selection import chi2
from keras.datasets import imdb
from time import time
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import f1_score
from sklearn.metrics.pairwise import cosine_similarity
from tmu.models.autoencoder.autoencoder import TMAutoEncoder

target_words = ['awful', 'terrible', 'lousy', 'abysmal', 'crap', 'outstanding', 'brilliant', 'excellent', 'superb', 'magnificent', 'marvellous', 'truck', 'plane', 'car', 'cars', 'motorcycle',  'scary', 'frightening', 'terrifying', 'horrifying', 'funny', 'comic', 'hilarious', 'witty']

clause_weight_threshold = 0
number_of_examples = 5
factor = 1
clauses = factor*20
T = factor*40
s = 5.0
accumulation = 4

# clause_weight_threshold = 0
# number_of_examples = 2000
# factor = 4
# clauses = factor*20
# T = factor*40
# s = 5.0
# accumulation = 24

print("Number of clauses:", clauses)

NUM_WORDS=10000
INDEX_FROM=2

Number of clauses: 20


**Keras Dataset**  
This is a dataset of 25,000 movies reviews from IMDB, labeled by sentiment (positive/negative). Reviews have been preprocessed, and each review is encoded as a list of word indexes (integers). For convenience, words are indexed by overall frequency in the dataset, so that for instance the integer "3" encodes the 3rd most frequent word in the data. This allows for quick filtering operations such as: "only consider the top 10,000 most common words, but eliminate the top 20 most common words"

In [10]:
print("Downloading dataset of users' reviews for movies...")
train,test = keras.datasets.imdb.load_data(num_words=NUM_WORDS, index_from=INDEX_FROM)

train_x,train_y = train
test_x,test_y = test
print("the number of reviews in train set =",train_x.size)
print("the first positive review of trainging (index from 1) has the following words indeces:",train_x[0])
print("you can see there is word teenge which has the integer 1621 ordered by frequency")
print(len(train_x[0]))

the number of reviews in train set = 25000
the first positive review of trainging (index from 1) has the following words indeces: [1, 13, 21, 15, 42, 529, 972, 1621, 1384, 64, 457, 4467, 65, 3940, 3, 172, 35, 255, 4, 24, 99, 42, 837, 111, 49, 669, 2, 8, 34, 479, 283, 4, 149, 3, 171, 111, 166, 2, 335, 384, 38, 3, 171, 4535, 1110, 16, 545, 37, 12, 446, 3, 191, 49, 15, 5, 146, 2024, 18, 13, 21, 3, 1919, 4612, 468, 3, 21, 70, 86, 11, 15, 42, 529, 37, 75, 14, 12, 1246, 3, 21, 16, 514, 16, 11, 15, 625, 17, 2, 4, 61, 385, 11, 7, 315, 7, 105, 4, 3, 2222, 5243, 15, 479, 65, 3784, 32, 3, 129, 11, 15, 37, 618, 4, 24, 123, 50, 35, 134, 47, 24, 1414, 32, 5, 21, 11, 214, 27, 76, 51, 4, 13, 406, 15, 81, 2, 7, 3, 106, 116, 5951, 14, 255, 3, 2, 6, 3765, 4, 722, 35, 70, 42, 529, 475, 25, 399, 316, 45, 6, 3, 2, 1028, 12, 103, 87, 3, 380, 14, 296, 97, 31, 2070, 55, 25, 140, 5, 193, 7485, 17, 3, 225, 21, 20, 133, 475, 25, 479, 4, 143, 29, 5534, 17, 50, 35, 27, 223, 91, 24, 103, 3, 225, 64, 15, 37, 1333, 87

In [11]:
print("Now let's get the words vocabulory and producing bit representation ...")
print("First downloading and build the dictionary in paython ...")
word_to_id = keras.datasets.imdb.get_word_index()
word_to_id = {k:(v+INDEX_FROM) for k,v in word_to_id.items()}

# "0" does not stand for a specific word, but instead is used to encode the pad token.
word_to_id["<PAD>"] = 0
word_to_id["<START>"] = 1
word_to_id["<UNK>"] = 2

print("Replace the dictionary value <=> id ...")
# Producing bit representation by replacing each index in review list with corrosponding word
# first we reverse the list (key,value => value,key)
id_to_word = {value:key for key,value in word_to_id.items()}

Now let's get the words vocabulory and producing bit representation ...
First downloading and build the dictionary in paython ...
Replace the dictionary value <=> id ...


In [12]:
training_documents = []
# .shape means convert the array and take only o index which is 25k iteratation
for i in range(train_y.shape[0]):
    terms = []
    for word_id in train_x[i]:
        # at each review extract the words that have
        terms.append(id_to_word[word_id].lower())
              
    # now training_document is list of reviews in words form
    training_documents.append(terms)

print(training_documents[65])

# do tha same for testing
testing_documents = []
for i in range(test_y.shape[0]):
    terms = []
    for word_id in test_x[i]:
        terms.append(id_to_word[word_id].lower())

    testing_documents.append(terms)

['<start>', 'dressed', 'to', 'kill', 'understandably', 'made', 'a', 'bit', 'of', 'a', '<unk>', 'when', 'first', 'released', 'in', '1980', 'you', 'had', 'police', 'girl', 'in', 'a', 'role', 'that', 'was', 'mega', 'erotic', 'as', 'angie', '<unk>', 'played', 'a', 'sexually', 'frustrated', 'housewife', 'looking', 'for', 'good', 'times', 'in', 'all', 'the', 'wrong', '<unk>', 'and', 'there', 'into', 'apartment', '<unk>', 'plus', 'nancy', 'allen', 'as', 'a', 'call', 'girl', 'michael', 'caine', 'as', 'norman', '<unk>', 'sophisticated', 'new', 'york', 'cousin', 'and', 'enough', 'lurid', 'imagery', 'to', 'last', 'two', 'movies', 'of', 'the', 'period', 'today', "it's", 'slightly', 'less', '<unk>', 'by', 'standards', 'and', 'such', 'though', 'the', '<unk>', 'version', 'has', 'some', 'of', 'the', 'most', 'hot', 'content', 'of', 'any', 'of', 'de', "palma's", 'films', 'at', 'least', 'in', 'his', 'quasi', 'auteur', 'period', 'of', 'the', '70s', 'and', 'early', '80s', 'where', 'he', 'seemed', 'to', 're

In [13]:
#represent the data as matrix of wordCounter
#https://towardsdatascience.com/basics-of-countvectorizer-e26677900f9c
#https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.CountVectorizer.html
#import pandas as pd

# get document encoding
def tokenizer(s):
    return s

vectorizer_X = CountVectorizer(tokenizer=tokenizer, lowercase=False, binary=True)
X_train = vectorizer_X.fit_transform(training_documents)
feature_names = vectorizer_X.get_feature_names_out()
number_of_features = vectorizer_X.get_feature_names_out().shape[0]
X_test = vectorizer_X.transform(testing_documents)

#Now the data for TM is X_train which is in the form of vectorize (each word in the Doc with the counter of use)
#print(number_of_features) = 9999
#df = pd.DataFrame(data=X_train.toarray(),columns = feature_names)
#print(df)

In [14]:
#prepare the target words
output_active = np.empty(len(target_words), dtype=np.uint32)
for i in range(len(target_words)):
    target_word = target_words[i]
    target_id = vectorizer_X.vocabulary_[target_word]
    output_active[i] = target_id
    
print(output_active)

[ 803 8920 5407  218 2180 6362 1250 3223 8703 5487 5584 9244 6668 1437
 1477 5915 7738 3730 8924 4400 3762 1882 4294 9850]


In [18]:
tm = TMAutoEncoder(clauses, T, s, output_active, max_included_literals=3, accumulation=accumulation, feature_negation=False, platform='CPU', output_balancing=True)

print("\nAccuracy Over 2 Epochs:")
for e in range(2):
    start_training = time()
    tm.fit(X_train, number_of_examples=number_of_examples)
    stop_training = time()

    print("\nEpoch #%d\n" % (e+1))

    precision = []
    for i in range(len(target_words)):
        precision.append(tm.clause_precision(i, True, X_train, number_of_examples=500))

    recall = []
    print("Calculating recall\n")
    for i in range(len(target_words)):
        recall.append(tm.clause_recall(i, True, X_train, number_of_examples=500))

    print("Clauses\n")
    for j in range(clauses):
        print("Clause #%d " % (j), end=' ')
        for i in range(len(target_words)):
            print("%s:W%d:P%.2f:R%.2f " % (target_words[i], tm.get_weight(i, j), precision[i][j], recall[i][j]), end=' ')

        l = []
        for k in range(tm.clause_bank.number_of_literals):
            if tm.get_ta_action(j, k) == 1:
                if k < tm.clause_bank.number_of_features:
                    l.append("%s(%d)" % (feature_names[k], tm.clause_bank.get_ta_state(j, k)))
                else:
                    l.append("¬%s(%d)" % (feature_names[k-tm.clause_bank.number_of_features], tm.clause_bank.get_ta_state(j, k)))
        print(" ∧ ".join(l))

    profile = np.empty((len(target_words), clauses))
    for i in range(len(target_words)):
        weights = tm.get_weights(i)
        profile[i,:] = np.where(weights >= clause_weight_threshold, weights, 0)

    similarity = cosine_similarity(profile)

    print("\nWord Similarity\n")
    for i in range(len(target_words)):
        print(target_words[i], end=': ')
        sorted_index = np.argsort(-1*similarity[i,:])
        for j in range(1, len(target_words)):
            print("%s(%.2f) " % (target_words[sorted_index[j]], similarity[i,sorted_index[j]]), end=' ')
        print()

    print("\nTraining Time: %.2f" % (stop_training - start_training))


Accuracy Over 2 Epochs:

Epoch #1

Calculating recall

Clauses

Clause #0  awful:W-1:Pnan:Rnan  terrible:W-1:Pnan:Rnan  lousy:W-1:Pnan:Rnan  abysmal:W-1:Pnan:Rnan  crap:W-1:Pnan:Rnan  outstanding:W-1:Pnan:Rnan  brilliant:W-1:Pnan:Rnan  excellent:W-1:Pnan:Rnan  superb:W-1:Pnan:Rnan  magnificent:W-1:Pnan:Rnan  marvellous:W-1:Pnan:Rnan  truck:W-1:Pnan:Rnan  plane:W-1:Pnan:Rnan  car:W-1:Pnan:Rnan  cars:W-1:Pnan:Rnan  motorcycle:W-1:Pnan:Rnan  scary:W-1:Pnan:Rnan  frightening:W-1:Pnan:Rnan  terrifying:W-1:Pnan:Rnan  horrifying:W-1:Pnan:Rnan  funny:W-1:Pnan:Rnan  comic:W-1:Pnan:Rnan  hilarious:W0:Pnan:R0.00  witty:W-1:Pnan:Rnan  '(128) ∧ '70s(128) ∧ '73(128) ∧ '80s(128) ∧ 'a(128) ∧ 'em(128) ∧ 'i(128) ∧ 'the(128) ∧ 00(128) ∧ 1(128) ∧ 100(128) ∧ 1000(128) ∧ 101(128) ∧ 11(128) ∧ 12(128) ∧ 13(128) ∧ 13th(128) ∧ 14(128) ∧ 15(128) ∧ 150(128) ∧ 16(128) ∧ 17(128) ∧ 18th(128) ∧ 19(128) ∧ 1930(128) ∧ 1930's(128) ∧ 1930s(128) ∧ 1931(128) ∧ 1932(128) ∧ 1934(128) ∧ 1936(128) ∧ 1938(128) ∧ 1939(128) ∧ 19

C:\Users\ahmedkk\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\tmu\models\autoencoder\autoencoder.py:386: RuntimeWarning: invalid value encountered in divide
  return 1.0 * true_positive / (true_positive + false_positive)


Calculating recall



C:\Users\ahmedkk\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\tmu\models\autoencoder\autoencoder.py:384: RuntimeWarning: invalid value encountered in divide
  return true_positive / (true_positive + false_negative)


Clauses

Clause #0  awful:W-1:Pnan:Rnan  terrible:W-1:Pnan:Rnan  lousy:W-1:Pnan:Rnan  abysmal:W-1:Pnan:Rnan  crap:W-1:Pnan:Rnan  outstanding:W-1:Pnan:Rnan  brilliant:W-1:Pnan:Rnan  excellent:W-1:Pnan:Rnan  superb:W-1:Pnan:Rnan  magnificent:W-1:Pnan:Rnan  marvellous:W-1:Pnan:Rnan  truck:W-1:Pnan:Rnan  plane:W-1:Pnan:Rnan  car:W-1:Pnan:Rnan  cars:W-1:Pnan:Rnan  motorcycle:W-1:Pnan:Rnan  scary:W-1:Pnan:Rnan  frightening:W-1:Pnan:Rnan  terrifying:W-1:Pnan:Rnan  horrifying:W-1:Pnan:Rnan  funny:W-1:Pnan:Rnan  comic:W-1:Pnan:Rnan  hilarious:W0:Pnan:R0.00  witty:W-1:Pnan:Rnan  '(128) ∧ '70s(128) ∧ '73(128) ∧ '80s(128) ∧ 'em(128) ∧ 00(128) ∧ 1(128) ∧ 101(128) ∧ 12(128) ∧ 13th(128) ∧ 14(128) ∧ 15(128) ∧ 150(128) ∧ 16(128) ∧ 17(128) ∧ 19(128) ∧ 1930(128) ∧ 1930's(128) ∧ 1930s(128) ∧ 1932(128) ∧ 1934(128) ∧ 1936(128) ∧ 1939(128) ∧ 1940(128) ∧ 1940's(128) ∧ 1940s(128) ∧ 1944(128) ∧ 1946(128) ∧ 1948(128) ∧ 1949(128) ∧ 1950(128) ∧ 1950's(128) ∧ 1953(128) ∧ 1955(128) ∧ 1957(128) ∧ 1958(128) ∧ 1960(128